In [2]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query, register_integrated_taxi_trips

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('storage_metrics')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

In [4]:
spark.sql("DESCRIBE DETAIL delta.`s3a://delta-data/gold/gold_integrated_taxi_trips`").show()

26/09/27 20:13:29 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/27 20:13:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|e8c2ca1f-b7a4-45f...|NULL|       NULL|s3a://delta-data/...|2026-09-27 19:08:...|2026-09-27 19:16:02|  [pickup_month]|               []|      19|  339275044|        {}|               1|               2|[appendOnly, inva...|
+------+--------------------+----+-----------+--------------------+---------

In [5]:
TABLES=[
    'air_quality_vs_demand',
    'average_trip_distance_by_weather',
    'monthly_zone_demand',
    'zone_weather_demand_variation'
]

In [7]:
for i in TABLES:
    spark.sql(f"DESCRIBE DETAIL delta.`s3a://delta-data/analytics/{i}`").show()


+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|dcf1cbf4-54c2-478...|NULL|       NULL|s3a://delta-data/...|2026-09-27 19:10:...|2026-09-27 19:10:09|              []|               []|       1|      39000|        {}|               1|               2|[appendOnly, inva...|
+------+--------------------+----+-----------+--------------------+---------

+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|2eafe9cc-64da-431...|NULL|       NULL|s3a://delta-data/...|2026-09-27 19:09:...|2026-09-27 19:28:31|              []|               []|       1|       3770|        {}|               1|               2|[appendOnly, inva...|
+------+--------------------+----+-----------+--------------------+---------

+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|fe3c2e0a-3539-439...|NULL|       NULL|s3a://delta-data/...|2026-09-27 19:09:...|2026-09-27 19:09:42|              []|               []|       1|      11490|        {}|               1|               2|[appendOnly, inva...|
+------+--------------------+----+-----------+--------------------+---------

+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
|format|                  id|name|description|            location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|       tableFeatures|
+------+--------------------+----+-----------+--------------------+--------------------+-------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+--------------------+
| delta|9090b99c-ac68-4a6...|NULL|       NULL|s3a://delta-data/...|2026-09-27 19:10:...|2026-09-27 19:28:56|              []|               []|       1|       7858|        {}|               1|               2|[appendOnly, inva...|
+------+--------------------+----+-----------+--------------------+---------

In [9]:
for i in TABLES:
    df = spark.read.format("delta").load(
        f"s3a://delta-data/analytics/{i}"
    )

    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(f"s3a://delta-data/analytics/csv/{i}")

    path = spark._jvm.org.apache.hadoop.fs.Path(f"s3a://delta-data/analytics/csv/{i}")
    fs = path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())

    csv_size = fs.getContentSummary(path).getLength()
    size = spark.sql(f"""
        DESCRIBE DETAIL delta.`s3a://delta-data/analytics/{i}`
    """).first()['sizeInBytes']
    print(i)
    print(f"size in csv: {csv_size}")
    print(f"size in delta: {size}")
    print('csv/delta')
    print(csv_size/size)
    print("---------")

26/09/19 13:48:31 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


air_quality_vs_demand
size in csv: 515092
size in delta: 38972
csv/delta
13.216976290670225
---------


26/09/19 13:48:32 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


average_trip_distance_by_weather
size in csv: 2929
size in delta: 3673
csv/delta
0.7974407841001906
---------


26/09/19 13:48:33 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


monthly_zone_demand
size in csv: 149221
size in delta: 11441
csv/delta
13.042653614194563
---------


26/09/19 13:48:34 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


zone_weather_demand_variation
size in csv: 40658
size in delta: 7736
csv/delta
5.255687693898656
---------
